In [ ]:
%pip install -q "mcp[cli]" python-dotenv openai

In [ ]:
import os
import asyncio

from dotenv import load_dotenv
from openai import OpenAI

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# Remote MCP Server
SERVER = "https://mcp.deepwiki.com/mcp"

# Your public GitHub repository
REPO = "amiteshks/2026_09_Gen_Agentic_AI"

print("MCP Server:", SERVER)
print("GitHub Repository:", REPO)

In [ ]:
load_dotenv(
    override=True,
    dotenv_path="../../.env.local"
)

my_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=my_api_key)

print("OpenAI client initialized")

In [ ]:
async def discover_tools():

    print(f"Connecting to {SERVER} ...")

    async with streamable_http_client(SERVER) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.list_tools()

            print("\nAvailable MCP Tools:\n")

            for tool in result.tools:
                print("Tool:", tool.name)
                print("Description:", tool.description)
                print("-" * 60)

            return result.tools


tools = await discover_tools()

In [ ]:
async def ask_deepwiki(question):

    async with streamable_http_client(SERVER) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            print("Calling DeepWiki MCP Server...")
            print("Repository:", REPO)
            print("Question:", question)
            print()

            response = await session.call_tool(
                "ask_question",
                {
                    "repoName": REPO,
                    "question": question
                }
            )

            # Check for MCP errors
            if response.is_error:
                print("DeepWiki returned an error.")
                print("Check whether the repository is indexed.")

            # Display response
            for block in response.content:
                if block.type == "text":
                    print(block.text)

            return response

In [ ]:
question = "What does this repository do?"

response = await ask_deepwiki(question)